# 32 — Participant-level multiple-instance learning

The selected lower-back model still makes both false-positive and false-negative errors, and notebook 31 showed that no probability threshold can remove both because healthy and stroke participant scores overlap. This experiment therefore changes the learning unit from a weakly labelled five-second window to the participant: all windows from one person form a bag and produce one participant-level loss.

## Predeclared experiment

1. Use only the Felius, Voisard, and Sint development sources. RevalExo and NONAN remain frozen and are not loaded.
2. Keep the primary modality fixed to lower-back acceleration magnitude.
3. Compare participant-level mean pooling (control) with gated-attention pooling (candidate). Training batches draw equal participant counts from every available source/class cell; participants with fewer than 16 windows are sampled with replacement only during training. Evaluation uses every real window available for each participant.
4. Tune training epochs only inside each outer training partition using a participant-disjoint inner split. Evaluate with three complete held-out-source tests and five fixed seeds.
5. Compare both MIL variants against the already selected deep ensemble using exactly matched source/seed evaluations. A candidate is accepted only if discrimination and calibration remain non-inferior, both mean FP and mean FN are non-increasing, and mean total errors fall by at least 10%.

This design follows the multiple-instance-learning principle that a set of observations can support one bag-level decision. Gated attention supplies an interpretable learned aggregation rather than assuming every gait window is equally informative. Wearable gait studies also commonly aggregate repeated events into participant- or session-level decisions while keeping test participants outside training. Relevant technical precedents: [attention-based deep MIL](https://proceedings.mlr.press/v80/ilse18a.html), [participant-level wearable aggregation](https://www.mdpi.com/1424-8220/22/18/6831), and [participant-aware gait MIL](https://www.mdpi.com/2076-3417/16/17/8354).


In [1]:
from pathlib import Path
import json, sys
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

module_path = ROOT / 'src/models/participant_attention_mil.py'
module_text = module_path.read_text(encoding='utf-8').lower()
for forbidden in ('revalexo_external_windows', 'nonan_external_windows'):
    assert forbidden not in module_text, f'Frozen signal reference found: {forbidden}'
assert torch.cuda.is_available(), 'CUDA GPU is required for this benchmark'
from src.models.participant_attention_mil import MILConfig, run
print('GPU:', torch.cuda.get_device_name(0))
print('Frozen external signal loaders referenced: NONE')


GPU: NVIDIA GeForce RTX 5060 Laptop GPU
Frozen external signal loaders referenced: NONE


In [2]:
mil = run(ROOT, MILConfig())


GPU: NVIDIA GeForce RTX 5060 Laptop GPU
source                label  
felius_2024           healthy     34
                      stroke     129
sint_maartenskliniek  healthy     20
                      stroke      10
voisard_2025          healthy     72
                      stroke      49
Name: group, dtype: int64


C:\Users\frank\Documents\MR-ICT Review Paper\src\models\participant_attention_mil.py:287: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  frame = frame.loc[frame.get("validation", False).fillna(False)]


Tuned felius_2024 mil_mean: 10 epochs


C:\Users\frank\Documents\MR-ICT Review Paper\src\models\participant_attention_mil.py:287: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  frame = frame.loc[frame.get("validation", False).fillna(False)]


Tuned felius_2024 mil_attention: 30 epochs


...

Complete voisard_2025 seed=314 mil_mean: BA=0.747 FP=7 FN=20


Complete voisard_2025 seed=314 mil_attention: BA=0.713 FP=12 FN=20


Complete voisard_2025 seed=515 mil_mean: BA=0.750 FP=11 FN=17


Complete voisard_2025 seed=515 mil_attention: BA=0.771 FP=11 FN=15


       method      held_out_source  auroc  brier  balanced_accuracy  specificity  sensitivity  false_positives  false_negatives
deep_ensemble          felius_2024 0.8588 0.1551             0.7637       0.7647       0.7628              8.0             30.6
deep_ensemble sint_maartenskliniek 0.8950 0.1387             0.8400       0.8000       0.8800              4.0              1.2
deep_ensemble         voisard_2025 0.9108 0.1338             0.8384       0.7583       0.9184             17.4              4.0
mil_attention          felius_2024 0.8383 0.1789             0.7479       0.6647       0.8310             11.4             21.8
mil_attention sint_maartenskliniek 0.7490 0.2445             0.7500       0.6000       0.9000              8.0              1.0
mil_attention         voisard_2025 0.8256 0.1856             0.7607       0.8111       0.7102             13.6             14.2
     mil_mean          felius_2024 0.8501 0.1306             0.7538       0.6471       0.8605           

In [3]:
display(mil['tuning'].round(4))
display(mil['summary'].round(4))
print(json.dumps(mil['decision'], indent=2))


,held_out_source,method,epoch,worst_balanced_accuracy,worst_specificity,worst_sensitivity,worst_auroc,mean_brier
0,felius_2024,mil_mean,10,0.8643,0.9286,0.8000,0.9071,0.0598
1,felius_2024,mil_mean,20,0.8643,0.9286,0.8000,0.9071,0.0626
2,felius_2024,mil_mean,30,0.8429,0.7857,0.9000,0.9571,0.0572
3,felius_2024,mil_mean,40,0.7000,1.0000,0.4000,0.9143,0.0918
4,felius_2024,mil_attention,10,0.8643,0.9286,0.8000,0.9143,0.0582
5,felius_2024,mil_attention,20,0.8643,0.9286,0.8000,0.9071,0.0569
6,felius_2024,mil_attention,30,0.8786,0.8571,0.9000,0.9429,0.0484
7,felius_2024,mil_attention,40,0.8643,0.9286,0.8000,0.9214,0.0569
8,sint_maartenskliniek,mil_mean,10,0.8286,0.8571,0.8000,0.8214,0.1248
9,sint_maartenskliniek,mil_mean,20,0.7286,0.8571,0.6000,0.8214,0.1615


,method,held_out_source,auroc,brier,balanced_accuracy,specificity,sensitivity,false_positives,false_negatives
0,deep_ensemble,felius_2024,0.8588,0.1551,0.7637,0.7647,0.7628,8.0,30.6
1,deep_ensemble,sint_maartenskliniek,0.8950,0.1387,0.8400,0.8000,0.8800,4.0,1.2
2,deep_ensemble,voisard_2025,0.9108,0.1338,0.8384,0.7583,0.9184,17.4,4.0
3,mil_attention,felius_2024,0.8383,0.1789,0.7479,0.6647,0.8310,11.4,21.8
4,mil_attention,sint_maartenskliniek,0.7490,0.2445,0.7500,0.6000,0.9000,8.0,1.0
5,mil_attention,voisard_2025,0.8256,0.1856,0.7607,0.8111,0.7102,13.6,14.2
6,mil_mean,felius_2024,0.8501,0.1306,0.7538,0.6471,0.8605,12.0,18.0
7,mil_mean,sint_maartenskliniek,0.8480,0.1627,0.7950,0.8100,0.7800,3.8,2.2
8,mil_mean,voisard_2025,0.8346,0.1797,0.7586,0.8111,0.7061,13.6,14.4


{
  "selected_method": "deep_ensemble",
  "candidate_decisions": {
    "mil_mean": {
      "accepted": false,
      "noninferior": false,
      "fp_and_fn_nonincreasing": true,
      "mean_total_errors": 21.333333333333332,
      "relative_total_error_reduction": 0.01840490797546022,
      "worst_source_deltas": {
        "balanced_accuracy_delta": -0.009986320109439228,
        "specificity_delta": -0.11127450980392162,
        "sensitivity_delta": -0.056668248694826895,
        "auroc_delta": -0.024243030545551547,
        "mean_brier_delta": 0.015156508128348706
      },
      "paired_bootstrap": {
        "balanced_accuracy": {
          "mean_delta": -0.04490662673338072,
          "ci_low": -0.06485827664399098,
          "ci_high": -0.02469593922840454
        },
        "specificity": {
          "mean_delta": -0.01828976034858388,
          "ci_low": -0.08868218954248365,
          "ci_high": 0.05405582788671017
        },
        "sensitivity": {
          "mean_delta": -0.07

In [4]:
pred = mil['predictions'].copy()
pred['prediction'] = (pred['probability'] >= 0.5).astype(int)
pred['error_type'] = 'correct'
pred.loc[(pred['y'] == 0) & (pred['prediction'] == 1), 'error_type'] = 'FP'
pred.loc[(pred['y'] == 1) & (pred['prediction'] == 0), 'error_type'] = 'FN'
display(pred.groupby(['method', 'source', 'error_type']).size().unstack(fill_value=0))
display(pred.groupby(['method', 'source'])[['windows', 'max_attention', 'effective_windows']].mean().round(3))


error_type                           FN  FP  correct
method        source                                
mil_attention felius_2024           109  57      649
              sint_maartenskliniek    5  40      105
              voisard_2025           71  68      466
mil_mean      felius_2024            90  60      665
              sint_maartenskliniek   11  19      120
              voisard_2025           72  68      465

windows  max_attention  effective_windows
method        source                                                         
mil_attention felius_2024           100.405          0.063             74.572
              sint_maartenskliniek  133.167          0.030            110.689
              voisard_2025           17.727          0.122             17.184
mil_mean      felius_2024           100.405          0.019            100.405
              sint_maartenskliniek  133.167          0.008            133.167
              voisard_2025           17.727          0.090             17.727

## Conclusion

Both participant-level candidates were rejected. Mean pooling changed mean source/seed errors from 21.73 to 21.33 (1.8% reduction), held mean FP at 9.8, and reduced mean FN from 11.93 to 11.53, but mean balanced accuracy fell from 0.8140 to 0.7691 and mean AUROC from 0.8882 to 0.8442. Its paired balanced-accuracy delta was -0.0449 (95% bootstrap interval -0.0649 to -0.0247). Gated attention increased errors to 23.33 and degraded discrimination and calibration. The selected lower-back deep ensemble therefore remains the baseline. No frozen external cohort was consulted in this decision.
